# 🏎️ Laboratorio di Reinforcement Learning: Missione Hill Climb Racing
### Progettazione della Funzione di Reward (Reward Engineering su Pendenze Collinari)

Benvenuti nel laboratorio di Reinforcement Learning su **Hill Climb Racing** (Box2D).  
In questa attività sperimentale progetterai la formulazione matematica della **funzione di Reward** per guidare un pilota artificiale (agente PPO) ad apprendere in modo autonomo a scalare colline sconnesse, dosare l'acceleratore sui dossi e superare il traguardo.

---
### 🎯 L'Obiettivo Didattico
Portare la vettura oltre gli **800 metri di distanza** su colline ripide e sconnesse, evitando:
1. 💥 **Lo schianto fatale del pilota**: la testa non deve mai impattare con il suolo (cappottamento).
2. ⏳ **Il blocco in salita (timeout)**: perdere slancio nel fondo valle e restare fermi senza trazione.
3. 🚀 **L'airtime incontrollato**: decollare sui dossi senza controllo dell'angolo di atterraggio.

---

## 1. L'idea: imparare per tentativi ed errori

Nel **Reinforcement Learning** (apprendimento per rinforzo) l'Intelligenza Artificiale non riceve istruzioni su *cosa* fare, ma impara da sola cercando di massimizzare un punteggio chiamato **Reward**. Ci sono due protagonisti:

| Protagonista | Chi è nel nostro laboratorio |
|---|---|
| **Agente** (chi impara) | Il "pilota" artificiale (algoritmo **PPO**) che comanda acceleratore e freno |
| **Environment** (il mondo) | Il simulatore fisico *Hill Climb Racing*: veicolo, pendenze procedurali, gravità e attrito |

Il ciclo di controllo si ripete **60 volte al secondo** (ogni *step* dura circa $0{,}016\text{ secondi}$):

```text
        ┌───────────────────  osservazione obs (8 sensori fisici)  ───────────────────┐
        ▼                                                                             │
   ┌──────────┐   azione action (0: folle, 1: gas avanti, 2: freno/retro)        ┌─────────────────────────┐
   │  AGENTE  │ ───────────────────────────────────────────────────────▶ │       ENVIRONMENT       │
   │ (pilota) │                                                          │ (veicolo + fisica Box2D)│
   └──────────┘ ◀─────────────  REWARD (premio + o penalità -)  ──────── └─────────────────────────┘
```

**Attenzione alla regola d'oro del Reinforcement Learning:**
L'agente non sa cosa significhi "guidare bene" o "scalare la collina". Conosce solo i numeri che gli dai tu! Il suo unico obiettivo matematico è **massimizzare la somma totale delle reward ricevute dall'inizio alla fine dell'episodio**:
$$R_{\text{totale}} = r_1 + r_2 + r_3 + \dots + r_T$$
Se scrivi una formula sbilanciata, l'agente troverà delle scorciatoie assurde (*Reward Hacking*) per fare punti senza scalare le colline (ad esempio stando fermo all'avvio per non rischiare di cappottarsi, oppure accelerando all'indietro)!

### 🛠️ Installazione e preparazione del motore di simulazione su Google Colab
Esegui le due celle seguenti per installare le librerie necessarie e scaricare il pacchetto didattico dalla repository ufficiale:  
👉 **https://github.com/EmanueleCiccarelli/RL_school**

In [ ]:
# 1. Installazione librerie di sistema e pacchetti Python su Colab
!apt-get update -qq
!apt-get install -y -qq swig
!pip install -q "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg tqdm rich matplotlib pygame

# 2. Download del pacchetto dalla repo GitHub ufficiale (se non già presente)
import os, sys
if not os.path.exists("play2learn_hillclimb"):
    !git clone -q https://github.com/EmanueleCiccarelli/RL_school.git
    if os.path.exists("RL_school"):
        %cd RL_school

In [ ]:
import play2learn_hillclimb
print("✅ Modulo play2learn_hillclimb caricato con successo su Google Colab!")

---
## 2. Conosci la Fisica della Scena e il Veicolo

Per progettare una buona funzione matematica devi conoscere le caratteristiche fisiche della vettura e del tracciato:

* **Punto di partenza (Spawn)**: La vettura parte ferma a quota $x = 0.0\text{ m}$ all'inizio della collina.
* **Obiettivo Traguardo**: Superare la distanza di sicurezza di **$800\text{ metri}$** di avanzamento orizzontale (`obs[0] >= 800.0`).
* **Due Ruote a Trazione Indipendente**: La vettura scarica a terra la potenza solo quando le ruote toccano il suolo (`rear_on == 1.0` o `front_on == 1.0`). In aria dare gas fa solo girare le ruote a vuoto e sbilancia il telaio per la conservazione del momento angolare!
* **A quale condizione si distrugge il veicolo?**
  1. **La testa del pilota è vulnerabile**: se tocca terra a seguito di un cappottamento o salto sbilanciato, l'episodio termina immediatamente con uno schianto irreversibile (`is_crashed = True`).
  2. **Perdita di slancio (Timeout)**: se la vettura scivola all'indietro o resta bloccata sul fondo di una valle ripida senza avanzare, l'episodio termina per timeout (`is_timeout = True`).

Esegui la cella seguente per visualizzare la **Mappa Fisica Annotata** del veicolo e del percorso:

In [ ]:
play2learn_hillclimb.mostra_mappa_ambiente()

---
## 3. Sensori (`obs`), Motori (`action`) ed Eventi di Fine Corsa

Ad ogni singolo step ($0{,}016\text{ s}$ / 60 FPS), la tua funzione `custom_reward(obs, action, is_success, is_crashed, is_timeout)` riceve:

### A. Il vettore `obs` con gli 8 sensori di bordo
| Indice | Sensore Fisico | Significato e Valori Tipici | Valore Ideale durante la Corsa |
| :---: | :--- | :--- | :--- |
| **`obs[0]`** | **Distanza X** | Avanzamento orizzontale in metri ($0 \to 1000\text{ m}$) | $\ge 800\text{ m}$ (traguardo) |
| **`obs[1]`** | **Quota Y** | Altezza del baricentro del veicolo rispetto al punto di spawn | Positiva (seguire il profilo) |
| **`obs[2]`** | **Velocità VX** | Velocità orizzontale in m/s ($> 0$ avanzamento, $< 0$ retromarcia) | $> 5.0\text{ m/s}$ costante |
| **`obs[3]`** | **Velocità VY** | Velocità verticale in m/s ($< 0$ salita, $> 0$ caduta/discesa) | Controllata sui salti |
| **`obs[4]`** | **Inclinazione Telaio** | Angolo di rollio in radianti ($0.0 = \text{piatto}$, $> 0$ impennata, $< 0$ muso giù) | Vicino a $0.0$ ($|\text{angle}| \le 0.4$) |
| **`obs[5]`** | **Velocità Angolare** | Velocità di rotazione del veicolo (rad/s) | $0.0$ (nessuna rotazione violenta) |
| **`obs[6]`** | **Ruota Posteriore** | $1.0$ se la ruota posteriore tocca terra, $0.0$ se è in aria | $1.0$ (piena trazione) |
| **`obs[7]`** | **Ruota Anteriore** | $1.0$ se la ruota anteriore tocca terra, $0.0$ se è in aria | $1.0$ (stabilità direzionale) |

### B. Il comando motore `action` scelto dall'agente
| Valore `action` | Azione del Pilota | Effetto Fisico sul Veicolo | Quando Usarlo |
| :---: | :--- | :--- | :--- |
| **`0`** | **Folle / Motore Spento (Idle)** | Il veicolo veleggia per pura inerzia senza alterare l'assetto | Nei salti alti per evitare la torsione di coppia |
| **`1`** | **Acceleratore (Gas)** | Applica coppia motrice in avanti alle ruote e fa impennare leggermente il muso | In salita e su tratti piani con ruote a terra |
| **`2`** | **Freno / Retromarcia (Brake)** | Frena con forza entrambe le ruote e abbassa il muso durante i decolli | Per correggere l'assetto in aria o evitare impennate |

### C. I 3 Flag Booleani di Fine Episodio
* **`is_success`** (`True`/`False`): Vittoria! La vettura ha superato il traguardo di **800 metri**.
* **`is_crashed`** (`True`/`False`): Schianto fatale! La testa del pilota ha urtato il suolo (cappottamento).
* **`is_timeout`** (`True`/`False`): Sono scaduti i tentativi di risalita: veicolo bloccato nel fondo valle senza slancio.

### 🔍 Esplorazione: guarda i numeri in tempo reale
Esegui la cella seguente per simulare 10 step con acceleratore attivo e osservare la telemetria dei sensori.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("HillClimbRacing-v0")
obs, _ = env.reset(seed=42)

nomi_azioni = {0: "0: Idle (Spento)  ", 1: "1: Gas (Avanti)   ", 2: "2: Freno (Brake)  "}

print("=" * 82)
print(" STEP | AZIONE             | DIST (m) | VX (m/s) | INCLINAZIONE | RUOTA POST | RUOTA ANT")
print("=" * 82)

for step in range(1, 11):
    action = 1  # Diamo gas!
    obs, reward, term, trunc, info = env.step(action)
    print(f" {step:4d} | {nomi_azioni[action]} | {obs[0]:7.2f}m | {obs[2]:6.2f}   | {obs[4]:10.2f} rad | {obs[6]:10.1f} | {obs[7]:9.1f}")

env.close()
print("=" * 82)

---
## 4. La Matematica dei Premi e le Trappole Classiche del Pilota 🧠

Prima di scrivere la tua funzione, tieni a mente questi **dati di riferimento**:
* 💥 **Ribaltamento violento (Schianto)**: può avvenire già dopo **60 – 120 step** ($1 - 2\text{ s}$) se si accelera troppo sui primi dossi.
* 🏆 **Corsa ideale di successo (800 metri)**: dura circa **500 – 900 step** ($8 - 15\text{ s}$).
* ⏳ **Timeout per stallo**: scatta dopo circa **1200 step** (o se la velocità rimane vicina a zero per troppi step).

Poiché la tua funzione `custom_reward` viene chiamata **ad ogni singolo step** (60 volte al secondo), piccoli premi o penalità si sommano centinaia di volte! Ecco le **2 trappole classiche del pilota**:

### ⚠️ Trappola 1: Il "Full Gas" (Ribaltamento sui Dossi)
* **L'errore:** Premi solo l'avanzamento (`dist_x` o `vx`) senza verificare se le ruote toccano terra.
* **Il calcolo dell'agente:** *"Se tengo il gas premuto al 100% guadagno punti subito. Non importa se sui dossi decollo: la reazione di coppia delle ruote che girano a vuoto mi fa ribaltare all'indietro atterrando sulla testa!"*
* **Come risolverla:** Premia la velocità `vx` **soltanto quando le ruote toccano il suolo** (`obs[6] > 0.5 or obs[7] > 0.5`), e penalizza fortemente le inclinazioni pericolose in volo (`abs(obs[4]) > 0.5`).

### ⚠️ Trappola 2: La "Prudenza Eccessiva" (Blocco nel Fondo Valle)
* **L'errore:** Metti una penalità di schianto enorme (`-500`) e nessuna spinta ad avanzare quando la collina si fa ripida.
* **Il calcolo dell'agente:** *"Se accelero sulle rampe ripide rischio di cappottarmi e prendere -500. Se mi fermo buono buono nel fondo valle a 40 metri prendo solo 0!"*
* **Come risolverla:** Assegna un premio progressivo proporzionale alla distanza percorsa (`dist_x * 0.02`), un premio per superare le creste e un bel bonus di vittoria (`+120.0`) per `is_success`!

---
## 5. Percorso Guidato a Livelli: Progetta la tua `custom_reward`

Ti proponiamo di costruire la tua funzione per gradi:
* **🥈 Livello 1 (Sopravvivenza e Avanzamento Base):** Gestisci gli eventi finali (`is_success`, `is_crashed`, `is_timeout`) e premia la velocità in avanti `vx` quando c'è trazione.
* **🥇 Livello 2 (Controllo Assetto e Anti-Cappottamento):** Aggiungi una penalità proporzionale all'inclinazione del telaio `abs(angle)` quando supera $0.4\text{ rad}$, contrastando le impennate.
* **💎 Livello 3 (Guida Esperta e Trazione Continua):** Premia la trazione combinata di entrambe le ruote (`rear_on` e `front_on`), penalizza la perdita di tempo a vuoto e incentiva la scalata delle colline più ripide.

In [ ]:
def custom_reward(obs, action, is_success, is_crashed, is_timeout):
    """
    Progetta qui la tua funzione matematica di Reward!

    SENSORI (obs):
      dist_x   = obs[0]  # Distanza percorsa in metri (0 -> 1000, traguardo >= 800)
      quota_y  = obs[1]  # Altezza del baricentro rispetto allo spawn
      vx       = obs[2]  # Velocità orizzontale (> 0 = avanti, < 0 = retro)
      vy       = obs[3]  # Velocità verticale (< 0 = salita, > 0 = discesa)
      angle    = obs[4]  # Inclinazione telaio in radianti (0 = orizzontale)
      ang_vel  = obs[5]  # Velocità angolare di rotazione (rad/s)
      rear_on  = obs[6]  # Ruota posteriore a contatto (1.0 = terra, 0.0 = aria)
      front_on = obs[7]  # Ruota anteriore a contatto (1.0 = terra, 0.0 = aria)

    AZIONE SCELTA (action):
      0 = Folle / Idle | 1 = Acceleratore (Gas) | 2 = Freno / Retromarcia

    EVENTI DI FINE EPISODIO (True / False):
      is_success : True se raggiungi almeno 800m
      is_crashed : True se la testa tocca terra (cappottamento fatale)
      is_timeout : True se resti bloccato senza slancio
    """
    dist_x = float(obs[0])
    quota_y = float(obs[1])
    vx = float(obs[2])
    vy = float(obs[3])
    angle = float(obs[4])
    ang_vel = float(obs[5])
    rear_on = float(obs[6])
    front_on = float(obs[7])

    reward = 0.0

    # =========================================================================
    # SCRIVI QUI LA TUA FORMULA DI REWARD SHAPING (Modifica e sperimenta!)
    # =========================================================================

    # 1. Eventi catastrofici e vittoria finale
    if is_crashed:
        return -100.0
    if is_timeout:
        return -40.0
    if is_success:
        return 120.0

    # 2. Premio per velocità con trazione attiva (almeno una ruota a terra)
    has_traction = (rear_on > 0.5 or front_on > 0.5)
    if has_traction:
        reward += max(0.0, vx * 1.5)
    else:
        reward -= 0.1

    # Piccolo incentivo progressivo per la distanza raggiunta
    reward += dist_x * 0.02

    # 3. Penalità per inclinazione pericolosa del telaio (anti-ribaltamento)
    if abs(angle) > 0.5:
        reward -= (abs(angle) - 0.5) * 4.0

    # Penalità per retromarcia ingiustificata
    if vx < -0.2:
        reward -= 0.5

    return float(reward)

print("✅ Funzione custom_reward salvata! Passa alla cella successiva per avviare l'addestramento.")

---
## 6. Addestramento PPO e Telemetria di Guida

Durante l'addestramento vedrai la telemetria di *rollout* con tre indicatori chiave:
* **`rollout/ep_rew_mean`**: La media della **tua custom reward** negli ultimi episodi.
* **`rollout/ep_len_mean`**: Quanti step dura in media una corsa (più è alto, più la vettura resiste senza schiantarsi).
* **`rollout/success_rate`**: La percentuale di volte in cui la vettura supera con successo gli **800 metri**.

Al termine verranno generati la **Dashboard con i grafici di telemetria**, la **Diagnosi del Driving Coach** e i **video MP4 dei percorsi migliori**!

In [ ]:
# Avvia l'addestramento PPO su 4 ambienti paralleli con la tua custom_reward
# Di default impostiamo 150.000 step: i primi tentativi superano 100-200 metri,
# mentre verso i 100k-150k step il pilota apprende a dosare il gas e superare gli 800 metri!
risultati = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="campagna_standard",
    total_timesteps=150_000,
    crea_gif_evoluzione=True,
    seed=42,
)

---
## 7. 🏆 Livelli Sfida Avanzati: Metti alla prova la Vettura su Tracciati Estremi!

Una volta calibrata la tua formula sul tracciato *Campagna*, metti alla prova il tuo pilota sui **3 percorsi speciali**:
* 🏜️ **`deserto_dune`**: Sabbia soffice a basso attrito e dune ondulate continue.
* ⛰️ **`montagna_estrema`**: Pendenze ripidissime con salite a $45^\circ$, burroni profondi e rocce ruvide.
* 🌑 **`luna_bassa_gravita`**: Gravità ridotta a $1/6$, decolli lunghissimi e alto rischio di cappottamento in volo!

In [ ]:
# Scegli la missione tra: "deserto_dune", "montagna_estrema", "luna_bassa_gravita"
risultati_sfida = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="deserto_dune",
    total_timesteps=150_000,
    crea_gif_evoluzione=True,
    seed=42,
)

---
## 8. 📊 Reportistica, Debug e Confronto tra Esperimenti

Quando sperimenti con la funzione di reward, hai a disposizione strumenti completi per analizzare, salvare e confrontare le prestazioni:

* ⭐ **I Tuoi Preferiti (`runs/favorites/`)**: È la tua galleria personale riservata **esclusivamente** ai modelli e ai video che decidi tu di salvare manualmente (`risultati.salva_preferito(...)`).
* 🎞️ **Timelapse Evoluzione (`risultati.mostra_evoluzione()`)**: Video riassuntivo che mostra la progressione dell'agente nelle varie fasi di training sullo stesso tracciato.
* 📄 **Scheda Tecnica Ufficiale (PDF, HTML, MD)**: Reportistica completa con metriche, grafici di telemetria e codice sorgente della formula usata.
* 📈 **Confronto tra Run**: Visualizza testa a testa su 20 scenari se le modifiche alla reward hanno aumentato la distanza media e le vittorie!

In [ ]:
# 1. Visualizza il timelapse dell'evoluzione dell'apprendimento nel tempo
risultati.mostra_evoluzione()

In [ ]:
# 2. Salva la run completata nei tuoi preferiti personali
risultati.salva_preferito(
    nome_personalizzato="Assetto_Colline_Stabile",
    note="Formula anti-ribaltamento calibrata: ottima trazione nelle valli ripide."
)

In [ ]:
# 3. Visualizza la galleria dei tuoi preferiti personali
play2learn_hillclimb.mostra_preferiti()

In [ ]:
# 4. Genera la Scheda Tecnica dell'esperimento (PDF vettoriale, HTML e Markdown)
# I file vengono salvati in una cartella dedicata per poter essere consultati e modificati
report = risultati.genera_report(
    autore="Nome Studente / Gruppo",
    titolo="Esperimento Hill Climb Racing",
    note="Inserisci qui le tue osservazioni sul comportamento dell'agente e sulle modifiche alla reward"
)

In [ ]:
# 5. Metti a confronto due o più run per analizzare l'impatto dei parametri
play2learn_hillclimb.confronta_run([risultati, risultati_sfida])

---
## 9. 🕒 Reportistica da Timestamp (Recupera Esperimenti Passati)

Hai eseguito più esperimenti nel corso della sessione e desideri rigenerare la **Scheda Tecnica** o rivedere i video di una run precedente?  
Non serve riaddestrare l'agente! Ti basta inserire il **timestamp** (o nome) della run che desideri analizzare.

### Come funziona:
1. **Consulta la cronologia**: Esegui la prima cella sottostante per visualizzare l'elenco di tutte le run salvate con i relativi timestamp (es. `20261116_101530`).
2. **Scegli il livello di dettaglio dal menu a tendina**:
   * **`Base`**: Genera istantaneamente la Scheda Tecnica ad alta definizione (PDF vettoriale, HTML, Markdown) e mostra i video già registrati per quella run.
   * **`Avanzata`**: Ricarica il modello addestrato ed esegue **nuove run di verifica**, registrando più video di test con telemetria approfondita!

In [ ]:
# 1. Visualizza la cronologia di tutti gli esperimenti per trovare il timestamp della run
play2learn_hillclimb.mostra_cronologia_run()

In [ ]:
# @title 📊 Generatore di Report da Timestamp { run: "auto" }
# @markdown Inserisci il **timestamp** della run passata (es. `20261116_101530` oppure `latest`) e seleziona il livello di reportistica:
timestamp_run = "latest"  # @param {type:"string"}
tipo_report = "Base"  # @param ["Base", "Avanzata (genera e salva più video e run)"]
autore = "Studente"  # @param {type:"string"}
note = ""  # @param {type:"string"}

# Genera il report per la run selezionata (Base o Avanzata)
report = play2learn_hillclimb.genera_report_da_timestamp(
    timestamp=timestamp_run,
    tipo=tipo_report,
    autore=autore,
    note=note
)

### 📈 Confronto Diretto tra Run Passate (tramite Timestamp)

Vuoi mettere a confronto due versioni della tua vettura per analizzare chi va più lontano?  
Inserisci i due timestamp nelle due celle seguenti per avviare il confronto testa a testa con **valutazione estesa su 20 gare** e **video split-screen affiancati**!

In [ ]:
# @title 🕒 1. Inserisci il Timestamp del Primo Esperimento (Run A) { run: "auto" }
# @markdown Inserisci il timestamp del primo esperimento (es. `20261116_101530` oppure `latest`):
timestamp_1 = "latest"  # @param {type:"string"}
print(f"✅ Primo esperimento impostato: {timestamp_1}")

In [ ]:
# @title 📊 2. Inserisci il Timestamp del Secondo Esperimento (Run B) e Avvia il Confronto { run: "auto" }
# @markdown Inserisci il timestamp del secondo esperimento (es. `20261116_094520` oppure `previous`):
timestamp_2 = "previous"  # @param {type:"string"}

# Unione automatica dei timestamp e avvio della valutazione estesa su 20 gare con video split-screen
timestamp_confronto = f"{timestamp_1.strip()}, {timestamp_2.strip()}"
play2learn_hillclimb.confronta_run(timestamp_confronto)

### 📦 Scarica l'Archivio degli Esperimenti (ZIP per Google Colab)

Vuoi scaricare sul tuo computer tutti i modelli addestrati, i grafici di telemetria, i report PDF e i video generati durante la sessione?  
Esegui la cella seguente per comprimere la cartella degli esperimenti in un file compresso `runs_hillclimb.zip` pronto per il download:

In [ ]:
# Comprime l'intera cartella delle run e avvia automaticamente il download su Google Colab
play2learn_hillclimb.comprimi_runs(nome_zip="runs_hillclimb.zip")